# T0 — Um remote de verdade: DagsHub (DVC + MLflow)

**Objetivo:** tirar o MLflow e o remote do DVC de dentro do Codespace. O CI/CD da T3 roda em outra máquina
(GitHub Actions) e não enxerga nada do que vive aqui.

```
            Aula 1                                 A partir de hoje
┌──────────────────────────┐          ┌──────────────┐       ┌──────────────────────┐
│ Codespace                │          │ Codespace    │ ────► │ DagsHub              │
│  MLflow  (mlflow.db)     │          │  código      │       │  MLflow (runs,       │
│  DVC     (../dvc-storage)│          │  pipeline    │       │          Registry)   │
└──────────────────────────┘          └──────────────┘       │  DVC    (dados,      │
                                              ▲              │          modelos)    │
                                      GitHub Actions ──────► └──────────────────────┘
```

| O quê | Aula 1 | A partir de hoje |
|---|---|---|
| Remote do DVC | pasta `../dvc-storage` | `https://dagshub.com/<usuário>/<repo>.dvc` |
| Servidor MLflow | `http://127.0.0.1:5000` (no Codespace) | `https://dagshub.com/<usuário>/<repo>.mlflow` |
| Quem alcança | só você, só neste Codespace | você, a API, o container e o GitHub Actions |

Esta tarefa é toda feita junto.

> 💡 As células `%%bash` rodam comandos de terminal. Os comandos que mudam o Git e o DVC vêm **comentados** (com `#`).
> Quando chegar a hora, tire o `#` (selecione as linhas e use **Ctrl + /**) e rode a célula.

In [1]:
import os, sys
from pathlib import Path

# Trabalha a partir da raiz do repositório (a pasta que contém o params.yaml)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "params.yaml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

## 0. A branch da aula

Hoje tudo acontece na branch `aula_02`: os commits, os pushes e o pipeline da T3. A `main` continua no estado
do fim da Aula 1 e só recebe o trabalho no último passo da T3, por um *pull request*.

In [2]:
%%bash
# git switch main
# git pull
# git switch -c aula_02
 git push -u origin aula_02
git branch --show-current

To https://github.com/anasegatelli/mlops-facens
   6f05c0a..c4763e3  aula_02 -> aula_02


branch 'aula_02' set up to track 'origin/aula_02'.
aula_02


## 1. Dependências de hoje

O FastAPI e o Uvicorn já vieram com o MLflow. Faltam as ferramentas de teste, que ficam num arquivo à parte:
elas não vão para a imagem de produção.

In [3]:
%%writefile requirements-dev.txt
pytest==9.2.0
httpx2==2.16.0

Overwriting requirements-dev.txt


In [1]:
%%bash
pip install -q -r requirements-dev.txt
pytest --version
docker --version


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


pytest 9.2.0
Docker version 29.10.1-1, build c48360f51c53d2029ab6721ad4d012f9e14df846


> 💬 Se `docker --version` der erro, veja "Problemas comuns" no `aula02/README.md` antes de seguir.

## 2. Ponto de partida

O script abaixo confere o repositório e completa o que estiver faltando: DVC inicializado, pipeline com os
5 estágios da T1 e dados v3 (2012 completo) versionados. Arraste o `recuperar_aula01.sh` para a pasta `scripts/`.

In [2]:
%%bash
bash scripts/recuperar_aula01.sh

1/4 DVC
    ok
2/4 Pipeline (dvc.yaml)
    dvc.yaml gravado com os 5 estágios da T1
3/4 Dados v3 (2012 completo)
Baixando de https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip ...
data/raw/bike.csv: 17,379 linhas | 2011-01-01 a 2012-12-31
4/4 Versão dos dados no DVC
    ok

Pronto. Dados: 17380 linhas | md5 7747b2ef


## 3. Conta e repositório no DagsHub

🖱️ No navegador:

1. Acesse **dagshub.com** e entre com **Sign up with GitHub**.
2. **Create +** → **New Repository** → terceiro cartão, **Connect a repository** → **GitHub**. Autorize o
   acesso e escolha o repositório da disciplina.
3. O repositório no DagsHub é um espelho do seu GitHub: cada `git push` aparece lá sozinho.
4. Copie o seu **token**: avatar (canto superior direito) → **Your Settings** → **Tokens**.

> 💬 O token é uma senha. Ele **não** pode aparecer em notebook, em código nem no Git.

## 4. O arquivo `.env`

As credenciais ficam num arquivo `.env`, fora do Git. A célula grava um modelo (`.env.example`), que pode ir
para o Git porque não tem segredo nenhum.

In [3]:
%%writefile .env.example
MLFLOW_TRACKING_URI=https://dagshub.com/SEU_USUARIO/SEU_REPOSITORIO.mlflow
MLFLOW_TRACKING_USERNAME=SEU_USUARIO
MLFLOW_TRACKING_PASSWORD=SEU_TOKEN

Writing .env.example


In [4]:
%%bash
grep -qx ".env" .gitignore || echo ".env" >> .gitignore   # primeiro o .gitignore...
[ -f .env ] || cp .env.example .env                        # ...depois o arquivo com o segredo
git check-ignore .env && echo "ok: o Git ignora o .env"

.env
ok: o Git ignora o .env


🖱️ Abra o **`.env`** no editor (não o `.env.example`) e troque `SEU_USUARIO`, `SEU_REPOSITORIO` e `SEU_TOKEN`
pelos seus valores. Salve e rode a célula abaixo, que carrega o arquivo e confere se ficou tudo preenchido.

In [7]:
from dotenv import load_dotenv

load_dotenv(".env", override=True)
uri = os.environ["MLFLOW_TRACKING_URI"]
assert "SEU_" not in open(".env").read(), "Ainda há um SEU_... no .env"
assert uri.endswith(".mlflow"), "A URI precisa terminar em .mlflow"

os.environ["DVC_REMOTE_URL"] = uri.removesuffix(".mlflow") + ".dvc"
print("MLflow:", uri)
print("DVC:   ", os.environ["DVC_REMOTE_URL"])
print("Token: ", "*" * 8 + os.environ["MLFLOW_TRACKING_PASSWORD"][-4:])

MLflow: https://dagshub.com/anasegatelli/mlops-facens.mlflow
DVC:    https://dagshub.com/anasegatelli/mlops-facens.dvc
Token:  ********9bc7


> 💬 As variáveis valem para este notebook. Em um terminal novo, carregue o `.env` com
> `set -a; source .env; set +a`.

## 5. DVC: o remote novo

O endereço do remote vai para o `.dvc/config` (Git). Usuário e token vão com `--local` para o
`.dvc/config.local`, que o DVC já deixa fora do Git.

In [8]:
%%bash
 dvc remote add -d -f origin "$DVC_REMOTE_URL"
 dvc remote modify origin --local auth basic
 dvc remote modify origin --local user "$MLFLOW_TRACKING_USERNAME"
 dvc remote modify origin --local password "$MLFLOW_TRACKING_PASSWORD"
cat .dvc/config
 dvc push

Setting 'origin' as a default remote.
[core]
    remote = origin
['remote "local"']
    url = ../../dvc-storage
['remote "origin"']
    url = https://dagshub.com/anasegatelli/mlops-facens.dvc
9 files pushed


🖱️ No DagsHub, abra o seu repositório: as pastas `data/` e `models/` agora mostram os arquivos que o DVC guarda.

## 6. MLflow: o servidor novo

O servidor do DagsHub começa vazio: sem experimentos e sem `bike-demand`. Rodamos o pipeline de novo para
registrar o treino lá. O `--force` é necessário porque nada mudou no código nem nos dados: mudou só o endereço
do MLflow, que o DVC não acompanha.

In [5]:
%%bash
 dvc repro --force evaluate plot

Verifying data sources in stage: 'data/raw/bike.csv.dvc'

Running stage 'prepare':
> python src/prepare.py
prepare: 17,379 linhas (0 removidas) e 14 colunas -> data/prepared/bike.csv

Running stage 'split':
> python src/split.py
split: treino 16,709 linhas | teste 670 linhas (2012-12-04 a 2012-12-31)

Running stage 'train':
> python src/train.py


2026/11/28 14:38:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'bike-demand' already exists. Creating a new version of this model...
2026/11/28 14:38:42 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: bike-demand, version 6
Created version '6' of model 'bike-demand'.


🏃 View run masked-cub-839 at: https://dagshub.com/anasegatelli/mlops-facens.mlflow/#/experiments/0/runs/28f1aaebbb024d3b9f49966ccf13222c
🧪 View experiment at: https://dagshub.com/anasegatelli/mlops-facens.mlflow/#/experiments/0
run_id=28f1aaebbb024d3b9f49966ccf13222c | bike-demand versão 6
Updating lock file 'dvc.lock'

Running stage 'evaluate':
> python src/evaluate.py
{
  "rmse": 62.89373227496592,
  "mae": 41.50570554415892,
  "r2": 0.8477560730724738
}
Updating lock file 'dvc.lock'

Running stage 'plot':
> python src/plot.py
plot: reports/previsto_vs_real.png salvo e anexado ao run 28f1aaebbb024d3b9f49966ccf13222c
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add .gitignore data/raw/bike.csv.dvc dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.


🖱️ No DagsHub: aba **Experiments** → **Go to MLflow UI**. O experimento `bike-demand-pipeline` e o modelo
`bike-demand` (versão 1) estão lá. A tag `data_version` fica no **run**, não na versão do modelo: abra o run
do experimento e veja as tags. O valor é o md5 dos dados v3 (`7747b2ef...`).

## 7. O primeiro `@champion` do servidor novo

Servidor novo, Registry novo: ainda não existe `@champion`. A versão que acabou de ser treinada assume o posto.

In [6]:
import json

import mlflow
from mlflow import MlflowClient
from bike import MODEL_NAME

client = MlflowClient()   # endereço e credenciais vêm das variáveis MLFLOW_TRACKING_*

versao = json.load(open("models/run.json"))["model_version"]
client.set_registered_model_alias(MODEL_NAME, "champion", versao)
client.set_model_version_tag(MODEL_NAME, versao, "validation_status", "approved")
client.set_model_version_tag(MODEL_NAME, versao, "motivo", "primeira versão no servidor remoto")
print(client.get_registered_model(MODEL_NAME).aliases)

/home/vscode/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'champion': '6'}


Com o `@champion` no lugar, o pipeline inteiro volta a fechar (quem fez o Desafio 1 tem o estágio `compare`,
que precisa dele):

In [7]:
%%bash
# dvc repro
dvc status

Data and pipelines are up to date.


## 8. Guardar

O `.env` fica de fora. Confira no `git status` antes do commit.

In [8]:
%%bash
git status --short
 git add .gitignore .env.example requirements-dev.txt scripts/recuperar_aula01.sh .dvc/config dvc.lock metrics.json reports
 git commit -m "Aula 2: remote do DVC e MLflow no DagsHub"
 dvc push
 git push

 M .dvc/config
 M .gitignore
 M aula02/t0_dagshub.ipynb
 M dvc.lock
?? .env.example
?? .vscode/
?? metrics.json
?? reports/previsto_vs_real.png
?? requirements-dev.txt
[aula_02 344ba08] Aula 2: remote do DVC e MLflow no DagsHub
 7 files changed, 66 insertions(+), 2 deletions(-)
 create mode 100644 .env.example
 create mode 100644 metrics.json
 create mode 100644 reports/previsto_vs_real.png
 create mode 100644 requirements-dev.txt
2 files pushed


To https://github.com/anasegatelli/mlops-facens
   c4763e3..344ba08  aula_02 -> aula_02
